Step 1 — Connect Google Drive and Verify Database Backup

In [1]:
from google.colab import drive
from pathlib import Path
import subprocess

# 1. Connect Google Drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

# 2. Locate the final embedding database backup
BACKUP_PATH = Path(
    "/content/drive/MyDrive/IBM_GraphRAG/"
    "postgres_backups/ibm_embeddings_final.dump"
)

print("STEP 1 — ENVIRONMENT CHECK")
print("=" * 50)

assert BACKUP_PATH.is_file(), "Final database backup not found!"

size_mb = BACKUP_PATH.stat().st_size / (1024 ** 2)

print("✅ Google Drive connected")
print("✅ Final database backup found")
print("Backup:", BACKUP_PATH.name)
print("Size:", round(size_mb, 2), "MB")

# 3. Check GPU availability
gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader"
    ],
    capture_output=True,
    text=True
)

if gpu.returncode == 0:
    print("✅ GPU:", gpu.stdout.strip())
else:
    print("⚠️ GPU not detected")

print("\n✅ Step 1 completed")

Mounted at /content/drive
STEP 1 — ENVIRONMENT CHECK
✅ Google Drive connected
✅ Final database backup found
Backup: ibm_embeddings_final.dump
Size: 28.34 MB
✅ GPU: Tesla T4, 15360 MiB

✅ Step 1 completed


Step 2 — PostgreSQL Setup and Database Restore

In [2]:
import subprocess
import sys
import importlib.util
from pathlib import Path

BACKUP = Path(
    "/content/drive/MyDrive/IBM_GraphRAG/"
    "postgres_backups/ibm_embeddings_final.dump"
)

DATABASE_URL = "postgresql:///ibm_rag?user=root"
MODEL = "qwen3-embedding:0.6b"

print("STEP 2 — POSTGRESQL SETUP")
print("=" * 50)

assert BACKUP.is_file(), "Final backup not found!"

# 1. Install PostgreSQL and pgvector
subprocess.run(["apt-get", "update", "-qq"], check=True)

subprocess.run([
    "apt-get", "install", "-y", "-qq",
    "postgresql", "postgresql-client",
    "postgresql-16-pgvector"
], check=True)

if importlib.util.find_spec("psycopg") is None:
    subprocess.run([
        sys.executable, "-m", "pip",
        "install", "-q", "psycopg[binary]"
    ], check=True)

import psycopg

# 2. Start PostgreSQL
subprocess.run(
    ["service", "postgresql", "start"],
    check=True
)

def admin_sql(sql):
    result = subprocess.run([
        "runuser", "-u", "postgres", "--",
        "psql", "-At", "-d", "postgres", "-c", sql
    ], capture_output=True, text=True, check=True)
    return result.stdout.strip()

# 3. Create PostgreSQL user if necessary
if not admin_sql(
    "SELECT 1 FROM pg_roles WHERE rolname = 'root'"
):
    admin_sql("CREATE ROLE root WITH LOGIN SUPERUSER")

# 4. Restore only if database does not exist
database_exists = admin_sql(
    "SELECT 1 FROM pg_database WHERE datname = 'ibm_rag'"
) == "1"

if not database_exists:
    print("Restoring database from final backup...")

    subprocess.run([
        "runuser", "-u", "postgres", "--",
        "createdb", "-O", "root", "ibm_rag"
    ], check=True)

    subprocess.run([
        "pg_restore",
        "--exit-on-error",
        "--no-owner",
        "--no-privileges",
        "-d", "ibm_rag",
        str(BACKUP)
    ], check=True)

    print("✅ Database restored")
else:
    print("✅ Existing database detected — no overwrite")

# 5. Verify database contents
with psycopg.connect(DATABASE_URL) as conn:

    documents = conn.execute(
        "SELECT COUNT(*) FROM documents"
    ).fetchone()[0]

    parents = conn.execute(
        "SELECT COUNT(*) FROM semantic_chunks"
    ).fetchone()[0]

    chunks = conn.execute(
        "SELECT COUNT(*) FROM retrieval_chunks_overlap_v1"
    ).fetchone()[0]

    embeddings = conn.execute("""
        SELECT COUNT(*)
        FROM chunk_embeddings
        WHERE model_name = %s
    """, (MODEL,)).fetchone()[0]

    index_status = conn.execute("""
        SELECT i.indisvalid
        FROM pg_index i
        WHERE i.indexrelid =
          to_regclass('public.chunk_embeddings_qwen_hnsw_idx')
    """).fetchone()

print("\nDATABASE STATUS")
print("=" * 50)
print("Documents:", documents)
print("Semantic chunks:", parents)
print("Retrieval chunks:", chunks)
print("Qwen embeddings:", embeddings)
print("HNSW index valid:", bool(index_status and index_status[0]))

assert (documents, parents, chunks, embeddings) == (
    496, 3200, 5098, 5098
), "Database contents do not match the expected final backup!"

assert index_status and index_status[0], "HNSW index missing or invalid!"

print("\n✅ PostgreSQL ready for Hybrid RAG")
print("✅ No embeddings regenerated")

STEP 2 — POSTGRESQL SETUP
Restoring database from final backup...
✅ Database restored

DATABASE STATUS
Documents: 496
Semantic chunks: 3200
Retrieval chunks: 5098
Qwen embeddings: 5098
HNSW index valid: True

✅ PostgreSQL ready for Hybrid RAG
✅ No embeddings regenerated


Step 3 — Ollama and Query Embedding Setup

In [3]:
import subprocess
import shutil
import time
import json
import urllib.request
from pathlib import Path

MODEL = "qwen3-embedding:0.6b"
OLLAMA_URL = "http://127.0.0.1:11434"

print("STEP 3 — OLLAMA + QWEN EMBEDDING")
print("=" * 50)

# 1. Install Ollama if needed
if not shutil.which("ollama"):
    subprocess.run(
        ["apt-get", "update", "-qq"], check=True
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "curl", "zstd"],
        check=True
    )
    subprocess.run(
        ["curl", "-fsSL", "https://ollama.com/install.sh",
         "-o", "/tmp/install_ollama.sh"],
        check=True
    )
    subprocess.run(
        ["bash", "/tmp/install_ollama.sh"],
        check=True
    )

print("✅ Ollama installed")

# 2. Check the Ollama server
def ollama_ready():
    try:
        with urllib.request.urlopen(
            f"{OLLAMA_URL}/api/tags", timeout=3
        ) as response:
            return response.status == 200
    except Exception:
        return False

# 3. Start the server if needed
if not ollama_ready():
    with open("/tmp/ollama.log", "a") as log:
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    for _ in range(45):
        if ollama_ready():
            break
        time.sleep(1)

assert ollama_ready(), "Ollama server is not running"
print("✅ Ollama server running")

# 4. Download Qwen embedding model
subprocess.run(
    ["ollama", "pull", MODEL],
    check=True
)
print("✅ Qwen embedding model ready")

# 5. Test embedding a user question
question = "How do I configure TLS in IBM MQ?"

payload = json.dumps({
    "model": MODEL,
    "input": question,
    "keep_alive": "15m"
}).encode("utf-8")

request = urllib.request.Request(
    f"{OLLAMA_URL}/api/embed",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(request, timeout=300) as response:
    result = json.load(response)

query_vector = result["embeddings"][0]

assert len(query_vector) == 1024

print("✅ Query embedding generated")
print("Vector dimensions:", len(query_vector))

# 6. Show model execution status
status = subprocess.run(
    ["ollama", "ps"],
    capture_output=True,
    text=True,
    check=True
)

print("\nOLLAMA MODEL STATUS:")
print(status.stdout.strip())

print("\n✅ Step 3 completed")

STEP 3 — OLLAMA + QWEN EMBEDDING
✅ Ollama installed
✅ Ollama server running
✅ Qwen embedding model ready
✅ Query embedding generated
Vector dimensions: 1024

OLLAMA MODEL STATUS:
NAME                    ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL               
qwen3-embedding:0.6b    ac6da0dfba84    1.2 GB    100% GPU     4096       llamacpp    14 minutes from now

✅ Step 3 completed


Step 4 — Dense Retrieval with pgvector

In [4]:
import json
import urllib.request
import psycopg

DATABASE_URL = "postgresql:///ibm_rag?user=root"
MODEL = "qwen3-embedding:0.6b"
OLLAMA_URL = "http://127.0.0.1:11434"


def dense_search(question, limit=5):

    # 1. Generate question embedding
    payload = json.dumps({
        "model": MODEL,
        "input": question
    }).encode("utf-8")

    request = urllib.request.Request(
        f"{OLLAMA_URL}/api/embed",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with urllib.request.urlopen(request, timeout=300) as response:
        vector = json.load(response)["embeddings"][0]

    assert len(vector) == 1024
    query_vector = json.dumps(vector)

    # 2. Search stored embeddings by cosine distance
    with psycopg.connect(DATABASE_URL) as conn:

        conn.execute("SET LOCAL hnsw.ef_search = 100")

        results = conn.execute("""
            SELECT
                c.optimized_id,
                c.document_id,
                d.title,
                c.content,
                1 - (e.embedding <=> %s::vector) AS similarity

            FROM chunk_embeddings e

            JOIN retrieval_chunks_overlap_v1 c
              ON c.optimized_id = e.optimized_id

            JOIN documents d
              ON d.document_id = c.document_id

            WHERE e.model_name = %s

            ORDER BY e.embedding <=> %s::vector

            LIMIT %s
        """, (query_vector, MODEL, query_vector, limit)).fetchall()

    return results


# 3. Test dense retrieval
question = "How do I configure TLS in IBM MQ?"

results = dense_search(question, limit=5)

print("DENSE SEARCH RESULTS")
print("=" * 55)
print("Question:", question)

for rank, (chunk_id, doc_id, title, content, similarity) in enumerate(results, 1):

    print(f"\nRANK {rank}")
    print("Document:", title)
    print("Chunk ID:", chunk_id)
    print("Cosine similarity:", round(similarity, 4))
    print("Text preview:", content[:350])
    print("-" * 55)

print("\nTotal retrieved chunks:", len(results))

DENSE SEARCH RESULTS
Question: How do I configure TLS in IBM MQ?

RANK 1
Document: IBM Configuring DataPower MQ client to use SSL in mutual authentication mode - United States
Chunk ID: e5ed1eb8-be3f-5b3d-ae2b-077a1d064824
Cosine similarity: 0.7384
Text preview: Title: IBM Configuring DataPower MQ client to use SSL in mutual authentication mode - United States

Text:
datapower; mq; websphere mq; ssl; mutual authentication mode; mq manager; 7993; 7993; 7993; machine type 7993 TECHNOTE (TROUBLESHOOTING)


-------------------------------------------------------

RANK 2
Document: IBM Configuring DataPower MQ client to use SSL in mutual authentication mode - United States
Chunk ID: 7690dbc0-55ca-545d-8f33-c4412d8d130d
Cosine similarity: 0.7344
Text preview: https://www.ibm.com/support/knowledgecenter/SS9H2Y_7.2.0/com.ibm.dp.doc/mq_securingwithansslproxyprofileobject.html [https://www.ibm.com/support/knowledgecenter/SS9H2Y_7.2.0/com.ibm.dp.doc/mq_securingwithansslproxyprofileobject.html] (Fi

Step 5 — BM25 Lexical Retrieval

This step implements BM25 as the lexical retrieval component of the Hybrid RAG pipeline.

The optimized IBM retrieval chunks are loaded from PostgreSQL and tokenized to build an in-memory BM25 index. For each user question, BM25 ranks chunks based on matching terms, term frequency, document frequency, and document length.

Unlike dense retrieval, BM25 does not use embeddings or vector similarity.



In [5]:
import subprocess
import sys
import re
import psycopg
import numpy as np
#install bm25 library
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "rank-bm25"],
    check=True
)

from rank_bm25 import BM25Okapi

DATABASE_URL = "postgresql:///ibm_rag?user=root"

# Convert text into searchable words
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


# 1. Load retrieval chunks from PostgreSQL
with psycopg.connect(DATABASE_URL) as conn:
    bm25_chunks = conn.execute("""
        SELECT
            c.optimized_id,
            c.document_id,
            d.title,
            c.content
        FROM retrieval_chunks_overlap_v1 c
        JOIN documents d
          ON d.document_id = c.document_id
        ORDER BY c.document_id, c.chunk_index
    """).fetchall()

# 2. Build BM25 index in RAM
tokenized_corpus = [
    tokenize(content)
    for _, _, _, content in bm25_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print("✅ BM25 index ready")
print("Indexed chunks:", len(bm25_chunks))


# 3. Define BM25 search
def bm25_search(question, limit=5):

    scores = bm25.get_scores(tokenize(question))
    top_indices = np.argsort(scores)[::-1][:limit]

    return [
        (*bm25_chunks[i], float(scores[i]))
        for i in top_indices
        if scores[i] > 0
    ]


# 4. Test with the same question as Dense Search
question = "How do I configure TLS in IBM MQ?"

results = bm25_search(question, limit=5)

print("\nBM25 SEARCH RESULTS")
print("=" * 55)
print("Question:", question)

for rank, (chunk_id, doc_id, title, content, score) in enumerate(results, 1):

    print(f"\nRANK {rank}")
    print("Document:", title)
    print("Chunk ID:", chunk_id)
    print("BM25 score:", round(score, 4))
    print("Text preview:", content[:350])
    print("-" * 55)

print("\nTotal retrieved chunks:", len(results))

✅ BM25 index ready
Indexed chunks: 5098

BM25 SEARCH RESULTS
Question: How do I configure TLS in IBM MQ?

RANK 1
Document: IBM 2035 MQRC_NOT_AUTHORIZED Connecting to WebSphere MQ from WebSphere Application Server via CLIENT Bindings - United States
Chunk ID: 061a9c30-6a40-5a1f-8395-1e11cd74c57b
BM25 score: 20.2144
Text preview: If you do not have a security exit that performs username and password authentication, then you should configure mutual SSL/TLS authentication on your Server Connection channel to cause the queue manager to require a trusted certificate is provided by the application server. To do this you set "SSL Authentication" to "Required" in the MQ Explorer o
-------------------------------------------------------

RANK 2
Document: IBM Configure IBM Content Collector to use SSL to communicate with IBM FileNet P8. - United States
Chunk ID: 894696ad-5606-570a-a2dc-0aea56c887f5
BM25 score: 18.8743
Text preview: Title: IBM Configure IBM Content Collector to use SSL to communic

Step 6 — Hybrid Retrieval Using Reciprocal Rank Fusion

This step combines dense vector retrieval and BM25 lexical retrieval using Reciprocal Rank Fusion (RRF).

Both methods independently retrieve candidate chunks. RRF then combines their rankings without directly comparing cosine similarity scores with BM25 scores.

The final output contains the highest-ranked chunks from the combined retrieval results.

In [6]:
QUESTION = "How do I configure TLS in IBM MQ?"

def hybrid_search(question, limit=5, candidates=30):
    dense_results = dense_search(question, limit=candidates)
    bm25_results = bm25_search(question, limit=candidates)

    fused = {}

    for method, results in [
        ("dense", dense_results),
        ("bm25", bm25_results)
    ]:
        for rank, row in enumerate(results, start=1):
            chunk_id = row[0]

            if chunk_id not in fused:
                fused[chunk_id] = {
                    "row": row,
                    "rrf_score": 0.0,
                    "dense_rank": None,
                    "bm25_rank": None
                }

            fused[chunk_id]["rrf_score"] += 1 / (60 + rank)
            fused[chunk_id][f"{method}_rank"] = rank

    ranked = sorted(
        fused.values(),
        key=lambda x: x["rrf_score"],
        reverse=True
    )

    return ranked[:limit]


results = hybrid_search(QUESTION, limit=5)

print("HYBRID SEARCH — DENSE + BM25 + RRF")
print("=" * 55)
print("Question:", QUESTION)

for position, item in enumerate(results, start=1):
    chunk_id, doc_id, title, content, _ = item["row"]

    print(f"\nRANK {position}")
    print("Document:", title)
    print("Dense rank:", item["dense_rank"])
    print("BM25 rank:", item["bm25_rank"])
    print("RRF score:", round(item["rrf_score"], 6))
    print("Text preview:", content[:350])
    print("-" * 55)

print("\nTotal retrieved chunks:", len(results))

HYBRID SEARCH — DENSE + BM25 + RRF
Question: How do I configure TLS in IBM MQ?

RANK 1
Document: IBM 2035 MQRC_NOT_AUTHORIZED Connecting to WebSphere MQ from WebSphere Application Server via CLIENT Bindings - United States
Dense rank: 11
BM25 rank: 3
RRF score: 0.029958
Text preview: Details of how you configure trust within WebSphere Application Server, and whether the application server sends a certificate to MQ for authentication, are in the following locations in the WebSphere Application Server information center:  * To create or modify an SSL Configuration containing the appropriate SSL/TLS configuration for connecting to
-------------------------------------------------------

RANK 2
Document: IBM 2035 MQRC_NOT_AUTHORIZED Connecting to WebSphere MQ from WebSphere Application Server via CLIENT Bindings - United States
Dense rank: 12
BM25 rank: 8
RRF score: 0.028595
Text preview: Details of how to configure the private certificate and trust of a MQ queue manager, and to enable SSL

### Step 7 — Document-Level Selection

This step selects distinct IBM source documents from the Hybrid RRF ranking.

Because multiple high-ranking retrieval chunks may belong to the same document, the selection process keeps the highest-ranked chunk from each document.

This provides a more diverse set of source documents for the next context expansion step.



In [7]:
QUESTION = "How do I configure TLS in IBM MQ?"

# Get more candidates before removing duplicate document IDs
ranked_chunks = hybrid_search(
    QUESTION,
    limit=80,
    candidates=50
)

selected_docs = []
seen_docs = set()

for item in ranked_chunks:
    chunk_id, doc_id, title, content, score = item["row"]

    if doc_id in seen_docs:
        continue

    seen_docs.add(doc_id)
    selected_docs.append(item)

    if len(selected_docs) == 3:
        break

print("TOP 3 UNIQUE DOCUMENTS — HYBRID SEARCH")
print("=" * 55)

for rank, item in enumerate(selected_docs, 1):
    chunk_id, doc_id, title, content, _ = item["row"]

    print(f"\nRANK {rank}")
    print("Document:", title)
    print("Dense rank:", item["dense_rank"])
    print("BM25 rank:", item["bm25_rank"])
    print("RRF score:", round(item["rrf_score"], 6))
    print("Best chunk:", content[:450])
    print("-" * 55)

print("\nUnique documents selected:", len(selected_docs))

TOP 3 UNIQUE DOCUMENTS — HYBRID SEARCH

RANK 1
Document: IBM 2035 MQRC_NOT_AUTHORIZED Connecting to WebSphere MQ from WebSphere Application Server via CLIENT Bindings - United States
Dense rank: 11
BM25 rank: 3
RRF score: 0.029958
Best chunk: Details of how you configure trust within WebSphere Application Server, and whether the application server sends a certificate to MQ for authentication, are in the following locations in the WebSphere Application Server information center:  * To create or modify an SSL Configuration containing the appropriate SSL/TLS configuration for connecting to MQ, see the following topic:
   SSL configurations [http://www14.software.ibm.com/webapp/wsbroker/r
-------------------------------------------------------

RANK 2
Document: IBM Configuring DataPower MQ client to use SSL in mutual authentication mode - United States
Dense rank: 5
BM25 rank: 25
RRF score: 0.027149
Best chunk: PROBLEM(ABSTRACT)
 The WebSphere DataPower MQ Client can be configured to use S

Step 8 — Semantic Parent Expansion
This step expands the retrieved child chunks into their corresponding semantic parent chunks using the source relationships stored in PostgreSQL.
For this demonstration, the two manually reviewed, relevant IBM documents are selected. Their full semantic parent texts are retrieved to provide richer context for answer generation.

In [8]:
import psycopg

DATABASE_URL = "postgresql:///ibm_rag?user=root"

# For this test, use the two relevant documents we reviewed
relevant_docs = selected_docs[:2]
expanded_parents = []

with psycopg.connect(DATABASE_URL) as conn:

    for item in relevant_docs:
        chunk_id, doc_id, title, content, _ = item["row"]

        # Find the semantic parents linked to this child chunk
        source_ids = conn.execute("""
            SELECT source_semantic_ids
            FROM retrieval_chunks_overlap_v1
            WHERE optimized_id = %s
        """, (chunk_id,)).fetchone()[0]

        assert source_ids, "No source parents found"

        # Retrieve the complete parent texts
        parents = conn.execute("""
            SELECT semantic_id, semantic_index, content
            FROM semantic_chunks
            WHERE document_id = %s
              AND semantic_id = ANY(%s::uuid[])
            ORDER BY semantic_index
        """, (doc_id, source_ids)).fetchall()

        assert parents, "Parent expansion failed"

        for parent_id, parent_index, parent_text in parents:
            expanded_parents.append({
                "document_id": doc_id,
                "title": title,
                "parent_id": parent_id,
                "parent_index": parent_index,
                "content": parent_text
            })

print("SEMANTIC PARENT EXPANSION")
print("=" * 55)

for i, parent in enumerate(expanded_parents, 1):
    print(f"\nPARENT {i}")
    print("Document:", parent["title"])
    print("Parent index:", parent["parent_index"])
    print("Length:", len(parent["content"]), "characters")
    print("Text preview:")
    print(parent["content"][:500])
    print("-" * 55)

print("\nTotal expanded parents:", len(expanded_parents))
print("✅ Full parent texts available in RAM")
print("✅ PostgreSQL unchanged")

SEMANTIC PARENT EXPANSION

PARENT 1
Document: IBM 2035 MQRC_NOT_AUTHORIZED Connecting to WebSphere MQ from WebSphere Application Server via CLIENT Bindings - United States
Parent index: 14
Length: 1839 characters
Text preview:
Details of how you configure trust within WebSphere Application Server, and whether the application server sends a certificate to MQ for authentication, are in the following locations in the WebSphere Application Server information center:  * To create or modify an SSL Configuration containing the appropriate SSL/TLS configuration for connecting to MQ, see the following topic:
   SSL configurations [http://www14.software.ibm.com/webapp/wsbroker/redirect?version=phil&product=was-base-dist&topic=c
-------------------------------------------------------

PARENT 2
Document: IBM Configuring DataPower MQ client to use SSL in mutual authentication mode - United States
Parent index: 1
Length: 709 characters
Text preview:
PROBLEM(ABSTRACT)
 The WebSphere DataPower MQ Clie

Step 9 — Grounded Answer Generation with Llama 3.2

This step uses Llama 3.2 (3B) to generate an answer based on the semantic parent chunks retrieved by the Hybrid RAG pipeline.
The retrieved IBM documentation is provided as context with numbered source references. The model is instructed to answer only using the available evidence and acknowledge missing information rather than inventing configuration steps.

In [9]:
import subprocess
import urllib.request
import json

LLM_MODEL = "llama3.2:3b"
OLLAMA_URL = "http://127.0.0.1:11434"

QUESTION = "How do I configure TLS in IBM MQ?"

print("STEP 9 — LLAMA ANSWER GENERATION")
print("=" * 55)

# 1. Download Llama
subprocess.run(
    ["ollama", "pull", LLM_MODEL],
    check=True
)

print("✅ Llama model ready")

# 2. Prepare retrieved context
assert expanded_parents, "No parent chunks available"

context = "\n\n".join(
    f"[{i}] Source: {parent['title']}\n"
    f"{parent['content']}"
    for i, parent in enumerate(expanded_parents, 1)
)

# 3. Ask Llama to answer using only the context
payload = json.dumps({
    "model": LLM_MODEL,
    "stream": False,
    "options": {"temperature": 0},
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a technical assistant for IBM documentation. "
                "Answer using ONLY the provided context. "
                "Cite supporting sources using [1], [2], etc. "
                "Do not invent configuration steps or facts. "
                "If the context is insufficient for a complete answer, "
                "clearly explain what is missing."
            )
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{QUESTION}\n\n"
                f"IBM documentation:\n{context}"
            )
        }
    ]
}).encode("utf-8")

request = urllib.request.Request(
    f"{OLLAMA_URL}/api/chat",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(request, timeout=600) as response:
    result = json.load(response)

answer = result["message"]["content"]

# 4. Display answer
print("\nQUESTION:")
print(QUESTION)

print("\nLLAMA ANSWER:")
print("=" * 55)
print(answer)

print("\nSOURCES:")
for i, parent in enumerate(expanded_parents, 1):
    print(f"[{i}] {parent['title']}")

print("\n✅ Answer generated")
print("✅ Nothing saved to PostgreSQL or Google Drive")

STEP 9 — LLAMA ANSWER GENERATION
✅ Llama model ready

QUESTION:
How do I configure TLS in IBM MQ?

LLAMA ANSWER:
Based on the provided IBM documentation, it appears that configuring TLS in IBM MQ involves several steps:

1. **Configuring trust within WebSphere Application Server**: To connect to MQ using CLIENT Bindings, you need to create or modify an SSL Configuration in WebSphere Application Server. This includes specifying the Cipher Suite/Cipher Spec that matches the SSLCIPH setting on the Server Connection channel.
2. **Enabling SSL/TLS on a client connection**: To enable SSL/TLS on a client connection, you need to choose which SSL Configuration to use. This can be done in the Connection Factories and Activation Specifications settings.
3. **Configuring MQ for mutual authentication**: To use SSL in mutual authentication mode with a Remote WebSphere MQ Manager, you need to configure the MQ queue manager with a key database and its associated password file. This can be done using t

Step 10 — Automatic Context Retrieval and Parent Expansion

This step automates context selection for the Hybrid RAG pipeline.

Dense search and BM25 retrieve relevant child chunks, and Reciprocal Rank Fusion (RRF) combines their rankings. The highest-ranked chunks from two distinct source documents are selected.

Each selected child chunk is mapped back to its semantic parent using the source_semantic_ids relationships stored in PostgreSQL.

Duplicate parent chunks are removed, and the resulting context is kept in memory for Llama answer generation.

In [10]:
import psycopg

DATABASE_URL = "postgresql:///ibm_rag?user=root"
QUESTION = "How do I configure TLS in IBM MQ?"

MAX_DOCUMENTS = 2
CHUNKS_PER_DOCUMENT = 2

# 1. Run Hybrid Search
ranked = hybrid_search(QUESTION, limit=60, candidates=40)

# 2. Select top distinct documents and their best child chunks
selected = {}

for item in ranked:
    chunk_id, doc_id, title, content, _ = item["row"]

    if doc_id not in selected:
        if len(selected) >= MAX_DOCUMENTS:
            continue
        selected[doc_id] = {"title": title, "chunks": []}

    if len(selected[doc_id]["chunks"]) < CHUNKS_PER_DOCUMENT:
        selected[doc_id]["chunks"].append(chunk_id)

# 3. Retrieve linked semantic parents
rag_context = []
seen_parents = set()

with psycopg.connect(DATABASE_URL) as conn:
    for doc_id, data in selected.items():

        for chunk_id in data["chunks"]:

            parents = conn.execute("""
                SELECT
                    s.semantic_id,
                    s.semantic_index,
                    s.content
                FROM retrieval_chunks_overlap_v1 c
                JOIN semantic_chunks s
                  ON s.document_id = c.document_id
                 AND s.semantic_id = ANY(c.source_semantic_ids)
                WHERE c.optimized_id = %s
                ORDER BY s.semantic_index
            """, (chunk_id,)).fetchall()

            for parent_id, parent_index, parent_text in parents:

                if (doc_id, parent_id) in seen_parents:
                    continue

                seen_parents.add((doc_id, parent_id))

                rag_context.append({
                    "title": data["title"],
                    "document_id": doc_id,
                    "parent_id": parent_id,
                    "parent_index": parent_index,
                    "content": parent_text
                })

# 4. Display results
print("AUTOMATIC HYBRID CONTEXT")
print("=" * 55)
print("Question:", QUESTION)
print("Selected documents:", len(selected))
print("Unique semantic parents:", len(rag_context))

for i, parent in enumerate(rag_context, 1):
    print(f"\nPARENT {i}")
    print("Document:", parent["title"])
    print("Parent index:", parent["parent_index"])
    print("Length:", len(parent["content"]))
    print("Preview:", parent["content"][:300])
    print("-" * 55)

print("\n✅ Context ready in RAM")
print("✅ No database changes")

AUTOMATIC HYBRID CONTEXT
Question: How do I configure TLS in IBM MQ?
Selected documents: 2
Unique semantic parents: 4

PARENT 1
Document: IBM 2035 MQRC_NOT_AUTHORIZED Connecting to WebSphere MQ from WebSphere Application Server via CLIENT Bindings - United States
Parent index: 14
Length: 1839
Preview: Details of how you configure trust within WebSphere Application Server, and whether the application server sends a certificate to MQ for authentication, are in the following locations in the WebSphere Application Server information center:  * To create or modify an SSL Configuration containing the a
-------------------------------------------------------

PARENT 2
Document: IBM 2035 MQRC_NOT_AUTHORIZED Connecting to WebSphere MQ from WebSphere Application Server via CLIENT Bindings - United States
Parent index: 13
Length: 727
Preview: Details of how to configure the private certificate and trust of a MQ queue manager, and to enable SSL security on a Server Connection channel, see the foll

### Step 11 — Structured Answer Generation and Evidence Verification

This step improves citation reliability by requesting structured output from Llama 3.2.

For each technical statement, the model must provide a source ID and an exact supporting quotation from the corresponding semantic parent chunk.

Python verifies that the cited source exists and that the quotation appears in the selected source. Only verified statements are displayed, with citation numbers added programmatically.



In [12]:
import json
import re
import urllib.request

LLM_MODEL = "llama3.2:3b"
OLLAMA_URL = "http://127.0.0.1:11434"

assert rag_context, "No retrieved context available!"

# 1. Prepare numbered sources
context = "\n\n".join(
    f"[SOURCE {i}]\n"
    f"Document: {parent['title']}\n"
    f"Parent index: {parent['parent_index']}\n"
    f"Content:\n{parent['content']}"
    for i, parent in enumerate(rag_context, 1)
)

# 2. Generate a grounded answer
system_prompt = """
You are an IBM technical documentation assistant.

Rules:
- Answer using only the provided IBM documentation.
- Do not invent steps, commands, or configuration values.
- Cite the source of every technical claim using [1], [2], etc.
- Only cite sources that actually support the claim.
- If information is insufficient, explicitly state what is missing.
- Answer clearly in English.
"""

user_prompt = f"""
Question:
{QUESTION}

Retrieved IBM documentation:
{context}

Write a helpful technical answer with inline source citations.
"""

payload = json.dumps({
    "model": LLM_MODEL,
    "stream": False,
    "options": {
        "temperature": 0,
        "num_predict": 768
    },
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
}).encode("utf-8")

request = urllib.request.Request(
    f"{OLLAMA_URL}/api/chat",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(request, timeout=600) as response:
    result = json.load(response)

rag_answer = result["message"]["content"]

# 3. Display the answer
print("AUTOMATIC HYBRID RAG ANSWER")
print("=" * 55)
print("Question:", QUESTION)

print("\nLLAMA ANSWER:")
print(rag_answer)

# 4. Check citation references
used_citations = {
    int(n) for n in re.findall(r"\[(\d+)\]", rag_answer)
}

valid_citations = set(range(1, len(rag_context) + 1))
invalid_citations = used_citations - valid_citations

print("\nCITATION CHECK")
print("=" * 55)
print("Available sources:", len(rag_context))
print("Citations used:", sorted(used_citations))

if not used_citations:
    print("⚠️ No inline citations found")
elif invalid_citations:
    print("⚠️ Invalid citations:", sorted(invalid_citations))
else:
    print("✅ All citation numbers refer to available sources")

print("\nSOURCE REFERENCES:")
for i, parent in enumerate(rag_context, 1):
    print(f"[{i}] {parent['title']} — Parent {parent['parent_index']}")

print("\n✅ Step 11 completed")

JSONDecodeError: Unterminated string starting at: line 3 column 20 (char 34)

### Step 12 — Improving Source Citations Through Prompt Engineering

This step improves the Llama generation prompt to reduce incorrect source citations.

The model receives the four previously retrieved semantic parent chunks, each with a fixed citation identifier. It is instructed to use only these identifiers, place citations directly after supported claims, and avoid inventing additional references.



In [14]:
import json
import re
import urllib.request

LLM_MODEL = "llama3.2:3b"

# Use the same four parent chunks
sources = "\n\n".join(
    f"[{i}] {parent['title']}\n{parent['content']}"
    for i, parent in enumerate(rag_context, 1)
)

system_prompt = """
You are a careful technical assistant for IBM documentation.

Use ONLY the retrieved source texts to answer the question.

STRICT RULES:
1. Never invent commands, configuration steps, or parameters.
2. Include a command ONLY if its complete syntax appears
   explicitly in the retrieved source text.
3. A URL, webpage name, or documentation link is NOT a command.
4. If a source points to another document, do not assume
   you know the contents of that document.
5. Put citations [1], [2], [3], or [4] immediately after
   the statements they support.
6. Do not create a References or Sources section.
7. Do not cite a source unless it supports the statement.
8. If the retrieved texts do not contain a complete
   step-by-step procedure, say so clearly.
9. It is better to give a short, incomplete but accurate
   answer than a detailed answer containing invented steps.

Answer in clear English.
"""

payload = json.dumps({
    "model": LLM_MODEL,
    "stream": False,
    "options": {
        "temperature": 0,
        "num_predict": 700
    },
    "messages": [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                f"Question:\n{QUESTION}\n\n"
                f"Available IBM sources:\n{sources}"
            )
        }
    ]
}).encode("utf-8")

request = urllib.request.Request(
    "http://127.0.0.1:11434/api/chat",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(request, timeout=600) as response:
    result = json.load(response)

answer = result["message"]["content"]

print("IMPROVED PROMPT — LLAMA ANSWER")
print("=" * 55)
print(answer)

# Simple citation check
citations = {
    int(n) for n in re.findall(r"\[(\d+)\]", answer)
}
valid_ids = set(range(1, len(rag_context) + 1))

print("\nCITATION CHECK")
print("Used citations:", sorted(citations))
print("Invalid citations:", sorted(citations - valid_ids))

IMPROVED PROMPT — LLAMA ANSWER
Unfortunately, the provided sources do not contain a complete step-by-step procedure for configuring TLS in IBM MQ. However, I can provide some general information and configuration steps based on the available sources.

To configure TLS in IBM MQ, you need to configure the private certificate and trust of the MQ queue manager, and enable SSL security on a Server Connection channel.

According to [2], to configure the certificates and trust of the MQ queue manager, you should refer to the following topic:

* Configuring SSL on queue managers [http://publib.boulder.ibm.com/infocenter/wmqv7/v7r1/index.jsp?topic=%2Fcom.ibm.mq.explorer.doc%2Fe_ssl_channels.htm]

To configure the Server Connection channel, used by applications running in the application server to connect to MQ, you should refer to the following topic:

* Configuring SSL channels [http://publib.boulder.ibm.com/infocenter/wmqv7/v7r1/topic/com.ibm.mq.explorer.doc/e_ssl_channels.htm]

Additionally

### Step 13 — End-to-End Hybrid RAG Pipeline

This step combines hybrid retrieval, semantic parent expansion, and grounded answer generation into a reusable function.

For each question, the pipeline uses Qwen3 embeddings and BM25 to retrieve candidate chunks, combines their rankings with RRF, selects the highest-ranked source documents, and retrieves the linked semantic parent chunks.

Llama 3.2 then generates an answer using only the retrieved context.



In [15]:
import json
import re
import urllib.request
import psycopg

def hybrid_rag(question, max_documents=2, chunks_per_document=2):

    # 1. Hybrid search: Dense + BM25 + RRF
    ranked = hybrid_search(
        question, limit=80, candidates=40
    )

    # 2. Select best chunks from distinct documents
    selected = {}

    for item in ranked:
        chunk_id, doc_id, title, _, _ = item["row"]

        if doc_id not in selected:
            if len(selected) >= max_documents:
                continue
            selected[doc_id] = {
                "title": title,
                "chunk_ids": []
            }

        if len(selected[doc_id]["chunk_ids"]) < chunks_per_document:
            selected[doc_id]["chunk_ids"].append(chunk_id)

    # 3. Expand child chunks into semantic parents
    parents = []
    seen = set()

    with psycopg.connect(DATABASE_URL) as conn:
        for doc_id, data in selected.items():
            for chunk_id in data["chunk_ids"]:

                rows = conn.execute("""
                    SELECT s.semantic_id, s.semantic_index, s.content
                    FROM retrieval_chunks_overlap_v1 c
                    JOIN semantic_chunks s
                      ON s.document_id = c.document_id
                     AND s.semantic_id = ANY(c.source_semantic_ids)
                    WHERE c.optimized_id = %s
                    ORDER BY s.semantic_index
                """, (chunk_id,)).fetchall()

                for parent_id, parent_index, content in rows:
                    if parent_id in seen:
                        continue

                    seen.add(parent_id)

                    parents.append({
                        "title": data["title"],
                        "parent_index": parent_index,
                        "content": content
                    })

    if not parents:
        print("No relevant context found.")
        return None

    # 4. Build numbered source context
    context = "\n\n".join(
        f"[{i}] Document: {p['title']}\n"
        f"Content:\n{p['content']}"
        for i, p in enumerate(parents, 1)
    )

    allowed_ids = ", ".join(
        f"[{i}]" for i in range(1, len(parents) + 1)
    )

    # 5. Generate answer with Llama
    system_prompt = f"""
You are a careful IBM technical documentation assistant.

Use ONLY the retrieved source texts.

Rules:
- Never invent commands, steps, or configuration values.
- Include commands only if their complete syntax is in the sources.
- A documentation URL is not a command.
- Do not assume you have read documents linked from the sources.
- Valid citation IDs are ONLY: {allowed_ids}.
- Put citations immediately after supported technical claims.
- Do not add a separate Sources or References section.
- If the sources are insufficient, clearly say so.
- Prefer an incomplete but accurate answer over invented details.

Answer clearly in English.
"""

    payload = json.dumps({
        "model": "llama3.2:3b",
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 700
        },
        "messages": [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": (
                    f"Question:\n{question}\n\n"
                    f"IBM sources:\n{context}"
                )
            }
        ]
    }).encode("utf-8")

    request = urllib.request.Request(
        "http://127.0.0.1:11434/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with urllib.request.urlopen(request, timeout=600) as response:
        result = json.load(response)

    answer = result["message"]["content"]

    # 6. Check citation numbers
    citations = {
        int(n) for n in re.findall(r"\[(\d+)\]", answer)
    }
    valid_ids = set(range(1, len(parents) + 1))

    # 7. Display results
    print("HYBRID RAG ANSWER")
    print("=" * 55)
    print("Question:", question)
    print("\nANSWER:\n")
    print(answer)

    print("\nRETRIEVAL SUMMARY")
    print("Selected documents:", len(selected))
    print("Expanded parents:", len(parents))
    print("Used citations:", sorted(citations))
    print("Invalid citations:", sorted(citations - valid_ids))

    print("\nSOURCE REFERENCES")
    for i, p in enumerate(parents, 1):
        print(
            f"[{i}] {p['title']} "
            f"— Parent {p['parent_index']}"
        )

    return {
        "question": question,
        "answer": answer,
        "sources": parents,
        "invalid_citations": sorted(citations - valid_ids)
    }


# Test with a NEW question
test_question = (
    "What causes MQRC_NOT_AUTHORIZED (2035) "
    "when connecting to IBM MQ?"
)

rag_result = hybrid_rag(test_question)

HYBRID RAG ANSWER
Question: What causes MQRC_NOT_AUTHORIZED (2035) when connecting to IBM MQ?

ANSWER:

MQRC_NOT_AUTHORIZED (2035) is caused when the user identifier specified in the Java System Property 'user.name' is not authorized to access the queue manager.

[1], [2], [3], [4], and [5]

RETRIEVAL SUMMARY
Selected documents: 2
Expanded parents: 5
Used citations: [1, 2, 3, 4, 5]
Invalid citations: []

SOURCE REFERENCES
[1] IBM 2035 MQRC_NOT_AUTHORIZED Connecting to WebSphere MQ from WebSphere Application Server via CLIENT Bindings - United States — Parent 0
[2] IBM 2035 MQRC_NOT_AUTHORIZED Connecting to WebSphere MQ from WebSphere Application Server via CLIENT Bindings - United States — Parent 1
[3] IBM 2035 MQRC_NOT_AUTHORIZED Connecting to WebSphere MQ from WebSphere Application Server via CLIENT Bindings - United States — Parent 2
[4] IBM Changes in the default user identifier between WebSphere MQ V7.0.1 classes for JMS and WebSphere MQ V7.1 classes for JMS - United States — Pare

## Evalutaion

Step 14 — Generate Evaluation Questions with Qwen

In [16]:
import subprocess
import psycopg
import random
import json
import re
import urllib.request
from collections import defaultdict

GENERATOR_MODEL = "qwen3:8b"
DATABASE_URL = "postgresql:///ibm_rag?user=root"

print("STEP 14 — QWEN EVALUATION QUESTIONS")
print("=" * 55)

# 1. Prepare Qwen3 for question generation
subprocess.run(
    ["ollama", "pull", GENERATOR_MODEL],
    check=True
)
print("✅ Qwen3 8B ready")

# 2. Load suitable semantic parents
with psycopg.connect(DATABASE_URL) as conn:
    source_rows = conn.execute("""
        SELECT
            s.document_id,
            s.semantic_id,
            d.title,
            s.content
        FROM semantic_chunks s
        JOIN documents d
          ON d.document_id = s.document_id
        WHERE LENGTH(s.content) BETWEEN 600 AND 2200
        ORDER BY s.document_id, s.semantic_index
    """).fetchall()

# 3. Sample one parent from each of 8 distinct documents
by_document = defaultdict(list)

for row in source_rows:
    by_document[row[0]].append(row)

rng = random.Random(42)
document_ids = rng.sample(
    list(by_document),
    k=min(8, len(by_document))
)

sampled_sources = [
    rng.choice(by_document[doc_id])
    for doc_id in document_ids
]

# 4. Generate one question per passage
eval_samples = []

for index, (doc_id, parent_id, title, content) in enumerate(
    sampled_sources, 1
):

    prompt = f"""
You are creating technical evaluation questions.

Read the IBM documentation passage below.

Generate exactly ONE specific technical question
whose answer can be found directly in the passage.

Rules:
- The question must be understandable on its own.
- Do not ask about facts absent from the passage.
- Do not invent information.
- Do not mention "the passage" or "the document".
- Output ONLY the question, ending with a question mark.

IBM documentation:
{content}
"""

    payload = json.dumps({
        "model": GENERATOR_MODEL,
        "prompt": prompt,
        "stream": False,
        "think": False,
        "options": {
            "temperature": 0.3,
            "num_predict": 160
        }
    }).encode("utf-8")

    request = urllib.request.Request(
        "http://127.0.0.1:11434/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with urllib.request.urlopen(request, timeout=600) as response:
        result = json.load(response)

    question = result["response"].strip()
    question = re.sub(
        r"<think>.*?</think>", "", question, flags=re.DOTALL
    ).strip()

    # Keep the generated question, not any extra explanation
    if "?" in question:
        question = question[:question.index("?") + 1]

    if not question.endswith("?") or len(question) < 15:
        print(f"⚠️ Question {index} skipped: invalid format")
        continue

    eval_samples.append({
        "question": question,
        "document_id": doc_id,
        "parent_id": parent_id,
        "title": title,
        "source_text": content
    })

    print(f"\nQUESTION {len(eval_samples)}")
    print("Source:", title[:100])
    print("Q:", question, flush=True)

print("\n" + "=" * 55)
print("Generated questions:", len(eval_samples))
print("✅ Questions and reference IDs kept in RAM")
print("✅ No evaluation files saved")

STEP 14 — QWEN EVALUATION QUESTIONS
✅ Qwen3 8B ready

QUESTION 1
Source: IBM Error TASK001X while upgrading Portal 8.0.0.1 to 8.0.0.1 CF14 during import-nodes sub-task - Uni
Q: What is the HTTP response code indicated in the exception message from the ConfigTrace.log?

QUESTION 2
Source: IBM Locating ITM Workspace Application Support Files for z/OS Agents - United States
Q: Which IBM OMEGAMON product for z/OS supports version 7.5.0?

QUESTION 3
Source: IBM Security Bulletin: Persistent cross-site scripting vulnerability in Process Admin Console affect
Q: What type of vulnerability does IBM Business Process Manager have according to the security bulletin?

QUESTION 4
Source: IBM Tivoli Netcool/OMINbus Java Gateway for ServiceNow (nco_g_servicenow-3_0) - United States
Q: What is the part number for the Netcool/OMNIbus V7.4.0 package for Linux?

QUESTION 5
Source: IBM Unable to connect using network services: WebGUI or ssh after upgrading WebSphere DataPower firm
Q: What command is used t

Step 15 — Retrieval Evaluation: Dense vs. BM25 vs. Hybrid RRF
This step evaluates Dense Retrieval, BM25, and Hybrid RRF using the eight synthetic questions generated by Qwen3.
Each generated question has a known source semantic parent. Retrieved child chunks are matched to that parent through the source_semantic_ids relationships stored in PostgreSQL.
The evaluation reports Hit@5, MRR@10, and single-reference nDCG@10 for each retrieval method.

In [17]:
import math
import psycopg

assert eval_samples, "No evaluation questions found!"

# 1. Load child-to-parent relationships into RAM
with psycopg.connect(DATABASE_URL) as conn:
    rows = conn.execute("""
        SELECT optimized_id, source_semantic_ids
        FROM retrieval_chunks_overlap_v1
    """).fetchall()

parents_by_child = {
    str(chunk_id): {str(pid) for pid in (parent_ids or [])}
    for chunk_id, parent_ids in rows
}


# 2. Find the first child linked to the reference parent
def reference_rank(chunk_ids, parent_id):
    target = str(parent_id)

    for rank, chunk_id in enumerate(chunk_ids[:10], 1):
        if target in parents_by_child.get(str(chunk_id), set()):
            return rank

    return None


# 3. Evaluate all three retrieval methods
evaluation_ranks = {
    "Dense": [],
    "BM25": [],
    "Hybrid": []
}

print("RETRIEVAL EVALUATION")
print("=" * 60)
print(f"{'Question':<10} {'Dense':<10} {'BM25':<10} {'Hybrid':<10}")
print("-" * 60)

for i, sample in enumerate(eval_samples, 1):
    question = sample["question"]
    parent_id = sample["parent_id"]

    # Retrieve 40 candidates per method
    dense_rows = dense_search(question, limit=40)
    bm25_rows = bm25_search(question, limit=40)

    dense_ids = [row[0] for row in dense_rows]
    bm25_ids = [row[0] for row in bm25_rows]

    # Apply the same RRF formula used in hybrid_search
    rrf_scores = {}

    for chunk_ids in (dense_ids, bm25_ids):
        for rank, chunk_id in enumerate(chunk_ids, 1):
            rrf_scores[chunk_id] = (
                rrf_scores.get(chunk_id, 0)
                + 1 / (60 + rank)
            )

    hybrid_ids = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )

    ranks = {
        "Dense": reference_rank(dense_ids, parent_id),
        "BM25": reference_rank(bm25_ids, parent_id),
        "Hybrid": reference_rank(hybrid_ids, parent_id)
    }

    for method, rank in ranks.items():
        evaluation_ranks[method].append(rank)

    def show_rank(rank):
        return str(rank) if rank is not None else "Not found"

    print(
        f"Q{i:<9} "
        f"{show_rank(ranks['Dense']):<10} "
        f"{show_rank(ranks['BM25']):<10} "
        f"{show_rank(ranks['Hybrid']):<10}",
        flush=True
    )


# 4. Calculate evaluation metrics
def calculate_metrics(ranks):
    n = len(ranks)

    hit5 = sum(
        r is not None and r <= 5
        for r in ranks
    ) / n

    mrr10 = sum(
        1 / r for r in ranks
        if r is not None and r <= 10
    ) / n

    # One known reference parent; count its first match
    ndcg10 = sum(
        1 / math.log2(r + 1)
        for r in ranks
        if r is not None and r <= 10
    ) / n

    return hit5, mrr10, ndcg10


print("\nEVALUATION SUMMARY")
print("=" * 60)
print(f"{'Method':<12} {'Hit@5':>10} {'MRR@10':>10} {'nDCG@10':>12}")

for method, ranks in evaluation_ranks.items():
    hit5, mrr10, ndcg10 = calculate_metrics(ranks)

    print(
        f"{method:<12} "
        f"{hit5:>10.3f} "
        f"{mrr10:>10.3f} "
        f"{ndcg10:>12.3f}"
    )

print("\nEvaluated questions:", len(eval_samples))
print("✅ Evaluation completed in RAM")
print("✅ No files or database changes")

RETRIEVAL EVALUATION
Question   Dense      BM25       Hybrid    
------------------------------------------------------------
Q1         6          3          1         
Q2         3          2          2         
Q3         4          1          2         
Q4         1          3          1         
Q5         1          1          1         
Q6         1          1          1         
Q7         2          1          1         
Q8         Not found  Not found  Not found 

EVALUATION SUMMARY
Method            Hit@5     MRR@10      nDCG@10
Dense             0.750      0.531        0.615
BM25              0.875      0.646        0.704
Hybrid            0.875      0.750        0.783

Evaluated questions: 8
✅ Evaluation completed in RAM
✅ No files or database changes


Step 16 — Investigate the Failed Question (Q8)

In [19]:
import re

# 1. Inspect Q8 and its original source
sample = eval_samples[7]

question = sample["question"]
target_parent = str(sample["parent_id"])
target_document = str(sample["document_id"])

print("Q8 — RETRIEVAL FAILURE ANALYSIS")
print("=" * 60)
print("Question:", question)
print("Source document:", sample["title"])

# 2. Check whether the reference parent has indexed children
linked_children = [
    chunk_id
    for chunk_id, parent_ids in parents_by_child.items()
    if target_parent in parent_ids
]

print("\nIndexed children linked to reference parent:",
      len(linked_children))

# 3. Show relevant parts of the original source
source_text = sample["source_text"]

matches = list(
    re.finditer(r"CVSS|Base Score", source_text, re.IGNORECASE)
)

print("\nCVSS-related text in original parent:")

if matches:
    for match in matches[:5]:
        start = max(0, match.start() - 100)
        end = min(len(source_text), match.end() + 160)
        print("\n...", source_text[start:end], "...")
else:
    print("No CVSS-related phrase found.")
    print(source_text[:700])

# 4. Search deeper: top 100 candidates
dense_rows = dense_search(question, limit=100)
bm25_rows = bm25_search(question, limit=100)

dense_ids = [str(row[0]) for row in dense_rows]
bm25_ids = [str(row[0]) for row in bm25_rows]

# 5. Combine rankings using RRF
rrf_scores = {}

for chunk_ids in (dense_ids, bm25_ids):
    for rank, chunk_id in enumerate(chunk_ids, 1):
        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0)
            + 1 / (60 + rank)
        )

hybrid_ids = sorted(
    rrf_scores,
    key=rrf_scores.get,
    reverse=True
)

# 6. Check reference parent position
def find_parent_rank(chunk_ids):
    for rank, chunk_id in enumerate(chunk_ids, 1):
        if target_parent in parents_by_child.get(chunk_id, set()):
            return rank
    return None

print("\nREFERENCE PARENT RANK")
print("=" * 60)

for method, chunk_ids in [
    ("Dense", dense_ids),
    ("BM25", bm25_ids),
    ("Hybrid", hybrid_ids)
]:
    rank = find_parent_rank(chunk_ids)
    print(f"{method}: {rank if rank else 'Not found'}")

# 7. Examine the leading retrieval results
print("\nTOP 3 DENSE RESULTS:")
for row in dense_rows[:3]:
    print("-", row[2][:110])

print("\nTOP 3 BM25 RESULTS:")
for row in bm25_rows[:3]:
    print("-", row[2][:110])



Q8 — RETRIEVAL FAILURE ANALYSIS
Question: What is the CVSS Base Score for the vulnerability described in the IBM documentation?
Source document: IBM Security Bulletin: Potential Privilege Escalation in WebSphere Application Server Admin Console (CVE-2017-1731) - United States

Indexed children linked to reference parent: 1

CVSS-related text in original parent:

... uthenticated remote attacker could exploit this vulnerability to possibly gain elevated privileges.
CVSS Base Score: 8.8
CVSS Temporal Score: See https://exchange.xforce.ibmcloud.com/vulnerabilities/134912 [https://exchange.xforce.ibmcloud.com/vulnerabilities/1349 ...

... ticated remote attacker could exploit this vulnerability to possibly gain elevated privileges.
CVSS Base Score: 8.8
CVSS Temporal Score: See https://exchange.xforce.ibmcloud.com/vulnerabilities/134912 [https://exchange.xforce.ibmcloud.com/vulnerabilities/134912] for the ...

... ttacker could exploit this vulnerability to possibly gain elevated privileges

### so the question provided by QWEN was not good